# Using pyDESeq2 for DEG analysis of TCGA LUAD data

### Import

In [1]:
import os
import pickle as pkl

import anndata as ad
import numpy as np
import pandas as pd
import sklearn as sk
import scipy as sc 
import matplotlib.pyplot as plt

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

In [2]:
# Set directories
data_dir = '/data'

In [ ]:
# Load counts
counts = pd.read_csv(os.path.join(data_dir,'TCGA-LUAD-counts_hg38.csv'), sep=',', index_col = 0) # downloaded via TCGAbiolinks (v2.24.3) using "STAR - Counts"

# Load meta data - clinical data
meta = pd.read_csv(os.path.join(data_dir, 'TCGA_clinicalData_lung.txt'), sep='\t') # downloaded from TCGA

# Load meta data - LOY status
y_status = pd.read_csv(os.path.join(data_dir,'TCGA_Y_status_lung.csv'), sep=',') # from Qi et al., 2023, DOI: 10.1016/j.cell.2023.06.006

# Filter Y Status for LUAD samples
y_status = y_status[y_status['Cohort'] == 'LUAD'] 

### Processing and filtering

In [ ]:
# Create a DataFrame from expression column names with full barcodes and case_id 
# Extract the case_id from the column names of the expression data (TCGA-XX-YYYY - first three segments of the barcode)
barcode_df = pd.DataFrame({
    'sample_barcode': counts.columns,
})
barcode_df['case_id'] = barcode_df['sample_barcode'].str.slice(0, 12)  # Extract TCGA-XX-YYYY

In [ ]:
# Merge barcode info with metadata on 'case_id'
merged_metadata = pd.merge(barcode_df, y_status, on='case_id', how='left') # left keeps all samples barcodes

In [ ]:
# filter patient metadata
# exclude na values
metadata_cleaned = merged_metadata.dropna(subset=['case_tcga_sample_id'])

# only include primary samples 
metadata_cleaned = metadata_cleaned[metadata_cleaned['sample_barcode'].str[13:15] == '01']

# include only samples with full LOY (=LOY) or ROY status (=WT)
metadata_cleaned = metadata_cleaned[metadata_cleaned['Y_status'].isin(['WT', 'LOY'])]

In [ ]:
# Ensure the sample barcodes in y status metadata align with the columns in counts df
filtered_sample_barcodes = metadata_cleaned['sample_barcode'].values

# Subset counts to only LUAD samples where we have Y status information
counts_filt_luad = counts.loc[:, counts.columns.isin(filtered_sample_barcodes)]

In [45]:
# Get sample barcode for ROY and LOY
loy_samples = metadata_cleaned.loc[metadata_cleaned['Y_status'] == 'LOY', 'sample_barcode'] 
roy_samples = metadata_cleaned.loc[metadata_cleaned['Y_status'] == 'WT', 'sample_barcode'] 

# Subset expression matrix for each group
loy_counts = counts_filt_luad.loc[:, counts_filt_luad.columns.isin(loy_samples)]
roy_counts = counts_filt_luad.loc[:, counts_filt_luad.columns.isin(roy_samples)]

In [ ]:
# Function to filter genes
def filter_genes(df, min_samples=25, min_counts=10):
    return df[df.apply(lambda x: (x >= min_counts).sum() >= min_samples, axis=1)]

# Apply filtering 
loy_counts_filt = filter_genes(loy_counts)
roy_counts_filt = filter_genes(roy_counts)

# Get a list of all genes that pass
passed_genes = list(set(loy_counts_filt.index) | set(roy_counts_filt.index))

# Create the filtered count matrix with all passing genes
counts_luad_qc = counts_filt_luad.loc[passed_genes]

# Save or display the filtered count matrix
counts_luad_qc.to_csv(os.path.join(output_dir, 'filt_qc_counts_tcga_luad.csv'))
print(counts_luad_qc.head())

# Print some statistics
print(f"Original number of genes: {counts_filt_luad.shape[0]}")
print(f"Number of genes after filtering: {counts_luad_qc.shape[0]}")
print(f"Number of genes passing in LOY group: {loy_counts_filt.shape[0]}")
print(f"Number of genes passing in ROY group: {roy_counts_filt.shape[0]}")

### Single Factor Analysis

In [ ]:
# prepare dataframe for DESeq2
# convert values to integers
count_df = counts_luad_qc.astype(int)

# transpose dataframe
counts_df_t = count_df.transpose()

# set the sample_barcode as the index
metadata_cleaned = metadata_cleaned.set_index('sample_barcode')

# Ensure the indices of counts and metadata match
metadata_cleaned = metadata_cleaned.reindex(counts_df_t.index)

print(counts_df_t.index)
print(metadata_cleaned.index)

In [ ]:
# Set the class 'DeseqDataSet', it has a count and a metadata argument, it is an AnnData object
# the design_factor specifies the columns used to compare samples
inference = DefaultInference(n_cpus=8)

dds_single_tcga_luad_Y = DeseqDataSet(
    counts=counts_df_t,
    metadata=metadata_cleaned,
    design_factors="Y_status",
    refit_cooks=True,
    inference=inference,
)

In [ ]:
# After dds was initialized we can run the deseq2() method to fit dispersion and LFCs
dds_single_tcga_luad_Y.deseq2()

### Statistical analysis after single-factor analysis
Now that dispersions and LFCs were fitted, we may proceed with statistical tests to
compute p-values and adjusted p-values for differential expresion.

In [54]:
# statistics - This is the role of the :class:`DeseqStats` class. It has a unique mandatory argument,
# ``dds``, which should be a *fitted* :class:`DeseqDataSet <pydeseq2.dds.DeseqDataSet>`object.

# It also has a set of optional keyword arguments (see the :doc:`API documentation
# </api/docstrings/pydeseq2.ds.DeseqStats>`), among which:
#
# - ``alpha``: the p-value and adjusted p-value significance threshold (``0.05``
#   by default),
# - ``cooks_filter``: whether to filter p-values based on cooks outliers
#   (``True`` by default),
# - ``independent_filter``: whether to perform independent filtering to correct
#   p-value trends (``True`` by default).

stat_res_tcga_luad_Y = DeseqStats(dds_single_tcga_luad_Y, inference=inference)

In [ ]:
# PyDESeq2 computes p-values using Wald tests. This can be done using the
# :meth:`summary() <DeseqStats.summary>` method, which runs the whole statistical
# analysis, cooks filtering and multiple testing adjustement included.  
# the results are stored in the ``results_df`` attribute (``stat_res.results_df``).
stat_res_tcga_luad_Y.summary()

In [ ]:
# Extract results (log2FoldChange and padj)
results_df_tcga = stat_res_tcga_luad_Y.results_df
results = results_df_tcga[['log2FoldChange', 'padj']]
results

In [ ]:
# Add -log10(padj) for the volcano plot
results['-log10(padj)'] = -np.log10(results['padj']) #it is WT vs LOY

# Reverse the log2FC for plotting - to have LOY vs WT
results['log2FoldChange'] = -results['log2FoldChange']